# 🐍 Master Capstone Project: Production Event-Driven Task Engine
> **Grand Capstone Project**: Integrates all 10 modules into an enterprise-grade, event-driven task processing system.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Project Overview
This capstone integrates:
1. **Core Language & Typing**: Type hints, dataclasses, pattern matching.
2. **Data Structures**: `deque` priority queues, sets for deduplication.
3. **OOP & Design Patterns**: Observer pattern (listeners), Strategy pattern (execution handlers), Custom Exceptions.
4. **Error Handling & Context Managers**: Safe execution with automatic logging and metrics tracking.

## 🏗️ 2. Domain Models & Type Specifications

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
import uuid

class TaskPriority(Enum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3

@dataclass(frozen=True)
class Task:
    task_type: str
    payload: dict
    priority: TaskPriority = TaskPriority.MEDIUM
    task_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    created_at: datetime = field(default_factory=datetime.now)

task = Task("EMAIL_ALERT", {"to": "ops@company.com", "msg": "Server CPU high"}, TaskPriority.HIGH)
print("Created Task model:", task)

## ⚙️ 3. The Event-Driven Task Processor Engine

In [ ]:
from collections import deque
import time

class TaskEngine:
    def __init__(self):
        self._queue = deque()  # O(1) FIFO queue
        self._handlers = {}    # Strategy pattern registry
        self._listeners = []   # Observer pattern listeners
        self._processed_ids = set() # Deduplication set
    
    def register_handler(self, task_type: str, handler_fn):
        self._handlers[task_type] = handler_fn
    
    def add_listener(self, listener_fn):
        self._listeners.append(listener_fn)
    
    def submit(self, task: Task):
        if task.priority == TaskPriority.HIGH:
            self._queue.appendleft(task) # Jump to front of queue!
        else:
            self._queue.append(task)
    
    def process_all(self):
        results = []
        while self._queue:
            task = self._queue.popleft()
            if task.task_id in self._processed_ids:
                continue # Skip duplicates
            
            handler = self._handlers.get(task.task_type)
            if not handler:
                print(f"[Error] No handler registered for '{task.task_type}'")
                continue
            
            res = handler(task.payload)
            self._processed_ids.add(task.task_id)
            results.append((task.task_id, res))
            
            # Notify observers:
            for listener in self._listeners:
                listener(task, res)
        return results

print("TaskEngine architecture initialized successfully!")

## 🚀 4. Executing the Complete System Pipeline

In [ ]:
# 1. Define Strategy Handlers:
def handle_email(payload):
    return f"Email sent to {payload['to']}"

def handle_invoice(payload):
    return f"Invoice #{payload['invoice_id']} generated for ${payload['amount']}"

# 2. Define Observer Audit Listener:
def audit_logger(task: Task, result: str):
    print(f" [AUDIT] Task {task.task_id} ({task.task_type}) processed -> {result}")

# 3. Wire and execute:
engine = TaskEngine()
engine.register_handler("EMAIL_ALERT", handle_email)
engine.register_handler("GENERATE_INVOICE", handle_invoice)
engine.add_listener(audit_logger)

# Submit normal task:
engine.submit(Task("GENERATE_INVOICE", {"invoice_id": 1001, "amount": 450.0}, TaskPriority.LOW))
# Submit high priority task (jumps ahead!):
engine.submit(Task("EMAIL_ALERT", {"to": "ceo@company.com"}, TaskPriority.HIGH))

print("Starting processing queue:")
results = engine.process_all()
print("\nAll tasks completed successfully:", results)

---
## 📋 5. Course Completion Summary

🎉 **Congratulations! You have completed the comprehensive Python master curriculum from Basics to Advanced Architecture.**

```
Curriculum Master Checklist:
 ✅ 01_Basics                         : Execution, types, operators, conditionals, loops, strings
 ✅ 02_Data_Structures                : Lists, tuples, dicts, sets, comprehensions, internals
 ✅ 03_Functions_and_Modularity       : Functions, *args/**kwargs, scope, functional, imports
 ✅ 04_Object_Oriented_Programming    : Classes, inheritance, MRO, ABCs, properties, dunders
 ✅ 05_File_IO_and_Error_Handling     : Text/binary IO, context managers, serialization, exceptions
 ✅ 06_Advanced_Language_Features     : Iterators, generators, decorators, dataclasses, itertools
 ✅ 07_Concurrency_and_Parallelism    : Threads, multiprocessing, asyncio, concurrent.futures
 ✅ 08_Metaprogramming_and_Internals  : Metaclasses, descriptors, PyMalloc, GC, GIL
 ✅ 09_Testing_Typing_and_Code_Quality: Type hints, unittest, pdb profiling, packaging
 ✅ 10_Design_Patterns_and_Projects   : Creational, structural, behavioral, capstone engine
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Complete Real-Time ATM Application (`ATM` Project)
> **Architecture Overview**:
> - **Domain Exceptions**: `DepositError`, `WithdrawError`, `InsufficientFundsError`.
> - **Business Logic**: State management for user balances, PIN authentication, and withdrawal caps.
> - **Menu Dispatcher**: Structural pattern matching (`match-case`) directing operations cleanly.

In [ ]:
# Complete Real-Time ATM Machine Project
class DepositError(Exception): pass
class WithdrawError(Exception): pass
class InsufficientFundsError(Exception): pass

class ATMSystem:
    def __init__(self, account_no, initial_balance=10000.0):
        self.account_no = account_no
        self.balance = initial_balance
        self.transaction_history = []

    def deposit(self, amount):
        if amount <= 0:
            raise DepositError(f"Deposit amount Rs. {amount} is invalid (must be > 0)")
        self.balance += amount
        self.transaction_history.append(f"DEPOSIT: +Rs. {amount:,.2f}")
        print(f"  [SUCCESS] Deposited Rs. {amount:,.2f}. Current Balance: Rs. {self.balance:,.2f}")

    def withdraw(self, amount):
        if amount <= 0:
            raise WithdrawError(f"Withdrawal amount Rs. {amount} is invalid (must be > 0)")
        if amount > self.balance:
            raise InsufficientFundsError(
                f"Insufficient Funds: Requested Rs. {amount:,.2f}, Available: Rs. {self.balance:,.2f}"
            )
        self.balance -= amount
        self.transaction_history.append(f"WITHDRAWAL: -Rs. {amount:,.2f}")
        print(f"  [SUCCESS] Dispensed Rs. {amount:,.2f}. Current Balance: Rs. {self.balance:,.2f}")

    def balance_enquiry(self):
        print(f"  [INFO] Account {self.account_no} Balance: Rs. {self.balance:,.2f}")

    def print_mini_statement(self):
        print("\n" + "=" * 45)
        print(f"        MINI STATEMENT - ACC {self.account_no}")
        print("=" * 45)
        for tx in self.transaction_history[-5:]:
            print(f"  {tx}")
        print("-" * 45)
        print(f"  Available Balance : Rs. {self.balance:,.2f}")
        print("=" * 45)

# Simulating an interactive session
atm = ATMSystem("SB-10928374", initial_balance=15000.0)

print("--- Simulating ATM Operations ---")
atm.balance_enquiry()
atm.deposit(5000.0)
atm.withdraw(7500.0)

# Demonstrating Exception Handling in ATM
try:
    atm.withdraw(25000.0)  # Exceeds available balance
except (WithdrawError, InsufficientFundsError) as err:
    print(f"  [ATM SECURITY REJECTION] {type(err).__name__}: {err}")

atm.print_mini_statement()

### 2. Standard 5-Step RDBMS Database Application (`PYTHON-MYSQL`, `PYTHON-ORACLE`)
> **The 5 Universal Steps of Database Communication**:
> 1. **Import Database Driver**: (`import sqlite3` / `import mysql.connector` / `import cx_Oracle`)
> 2. **Establish Connection**: `con = driver.connect(...)`
> 3. **Create Cursor Object**: `cur = con.cursor()`
> 4. **Execute SQL Queries**: `cur.execute(query)` / `cur.fetchall()` / `cur.commit()`
> 5. **Close Resources**: `cur.close()`, `con.close()`

In [ ]:
# 5-Step Database Communication Workflow using built-in SQLite3
import sqlite3

# Step 1 & 2: Connect to Database (creates in-memory DB for zero footprint)
con = sqlite3.connect(":memory:")
print("Step 1 & 2: Connected to database successfully.")

try:
    # Step 3: Create Cursor
    cur = con.cursor()
    print("Step 3: Cursor created.")

    # Step 4: Execute DDL & DML Queries
    cur.execute("""
        CREATE TABLE students (
            sno INTEGER PRIMARY KEY,
            sname TEXT NOT NULL,
            course TEXT NOT NULL,
            marks REAL
        )
    """)
    
    # Inserting Records
    student_records = [
        (101, "Aditya Rao", "Python Full Stack", 88.5),
        (102, "Bhavna Patel", "Data Science", 92.0),
        (103, "Chetan Kumar", "AI & ML", 76.5)
    ]
    cur.executemany("INSERT INTO students VALUES (?, ?, ?, ?)", student_records)
    con.commit()
    print("Step 4: Executed DDL table creation and inserted 3 student records.")

    # Querying & Fetching
    print("\n--- Querying Database Records (cur.fetchall()) ---")
    cur.execute("SELECT * FROM students WHERE marks >= 80")
    for row in cur.fetchall():
        print(f"  Roll #{row[0]} | Name: {row[1]:<15} | Course: {row[2]:<20} | Marks: {row[3]}")

finally:
    # Step 5: Close Resources
    cur.close()
    con.close()
    print("\nStep 5: Cursor and connection closed cleanly.")